# Experiment v5 — Isolation Phase (Jupyter server)

Compute-budget rebuild of v4: **200-scenario** portfolio (100 single + 100 dual), **100 visits**,
ratios **{2, 5, 10, 20, 50}**, deterministic **STOP_COMPUTE** budget (thread-local `ComputeMeter`)
instead of wall-clock — results no longer depend on CPU contention / threads.

v5 also collects **per-turn operator telemetry** (`turn_telemetry.csv` per leaf) to explain the
single-vs-dual gap (see `analysis/V5_HYPOTHESES.md`): H1 blend coherence, H2 repair disruption,
H3 elite-pool collapse, H4 operator credit, H5 action composition. Collection is observation-only.

**Built for long, restart-prone runs.** The server can restart during the 12+ hour job and lose
almost nothing — the whole pipeline is checkpointed:
- the portfolio build is skipped if already complete (`_portfolio_complete.txt`);
- a **random-init invariance check** and a **thread-count auto-tune** run once, then are reused
  (`_random_init_ok.txt`, `_tuned_threads.txt`);
- the isolation phase resumes per (agent, ratio, budget) **and** per (scenario, visit) — only
  the turns that never finished are recomputed (a partial-write tail from a crash is discarded).

**To resume after a restart: just re-run the launch cell (section 3).** It picks up where it left off.

**Prerequisites:** repo on branch `v5-tests`; JDK 8+; Python 3 + pip; run from `Tribes-master/`.
Linux classpath separator is `:` (Windows uses `;`).

In [ ]:
import os
# EDIT to the absolute path of the Tribes-master directory on the server:
REPO = os.path.expanduser('~/Bachelor_Thesis/Tribes-master')
os.chdir(REPO)
print('cwd =', os.getcwd()); print(os.listdir('.'))   # expect src/, lib/, analysis/ ...

## 0. Environment check

In [ ]:
!java -version
!python --version
!nproc                 # number of cores
!free -g | head -2     # RAM in GB (need ~750 free for the run heap)

## 1. Build test
Compiles the whole source tree to `out/`. Prints `BUILD OK` only on success — fix any compile
error before running anything else.

In [ ]:
!rm -rf out && mkdir -p out
!find src -name '*.java' > _srcs.txt
!javac -d out -cp "lib/*" @_srcs.txt && echo 'BUILD OK' || echo 'BUILD FAILED'

## 2. Smoke test (~10 s) — pipeline correctness
Verifies FastGameStateIO round-trip, the shared random initial sequence across agents AND budgets,
folder layout, CSV schema, and the operator telemetry (EMCTS/BB log zero crossover events;
Expansion logs them; counters sane). If this fails, do NOT launch the full run.

In [ ]:
!java -Xmx4g -cp "out:lib/*" core.data.exp.NewExperimentSmokeTestV5

## 3. Run v5 (background + log) — also the RESUME cell
Launches the full run detached so it survives a disconnect. **Re-run this same cell to resume**
after a server restart — finished work is skipped, an interrupted config continues per
(scenario, visit).

Args: `[threads|auto] [computeUnitsCsv]`. `auto` (default) runs the pre-run phases:
portfolio build → startup verify → **random-init invariance check** → **thread auto-tune** →
isolation. The chosen thread count is saved and reused on restart.

Heap is **750 GB** (dedicated server). Each in-flight task holds one GameState deep-copy.

In [ ]:
!nohup java -Xmx750g -cp "out:lib/*" core.data.exp.RunExperimentV5 auto > v5_run.log 2>&1 &
print('v5 started/resumed in background -> v5_run.log')

## 4. Monitor progress, checkpoints & thread use (re-run to refresh)
Shows the pre-run checkpoints, how many leaf CSVs / configs are done, the live thread/CPU use
(`NLWP` = OS threads, `%CPU` = total across them — should sit well above 100%), and the log tail.

In [ ]:
import subprocess
print('cores       :', subprocess.run(['nproc'], capture_output=True, text=True).stdout.strip())
!echo "portfolio   : $(test -f test_results/v5/_portfolio/_portfolio_complete.txt && echo COMPLETE || echo building/incomplete)"
!echo "pre-run     : random-init=$(test -f test_results/v5/_random_init_ok.txt && echo ok || echo pending)  tuned-threads=$(cat test_results/v5/_tuned_threads.txt 2>/dev/null || echo pending)"
!echo "leaf CSVs   : $(find test_results/v5 -name results.csv 2>/dev/null | wc -l) / 36 configs"
!echo "configs done: $(grep -c . test_results/v5/_completed.txt 2>/dev/null || echo 0) / 36"
print('--- in-progress leaf turn counts (each full leaf = 20000 turns x 20 burns) ---')
!for f in $(find test_results/v5 -name results.csv 2>/dev/null); do printf '  %-46s %s rows\n' "$f" "$(($(wc -l < "$f")-1))"; done | tail -6
print('--- run process: PID / threads (NLWP) / CPU% / elapsed ---')
!ps -o pid,nlwp,pcpu,etime,comm -C java 2>/dev/null || echo '(no java process running)'
!pgrep -af RunExperimentV5 || echo '(RunExperimentV5 not running - finished, not started, or between phases)'
print('--- last 30 log lines ---')
!tail -n 30 v5_run.log 2>/dev/null || echo '(no log yet)'

## 5. Analyse (after the run completes)
`analyze_v5_full.py` produces the SAME tables/figures as v4 (T1–T8, fig3–fig8, E1–E16) plus the
single-vs-dual hypothesis battery on the telemetry:

| analysis | hypothesis | question |
|---|---|---|
| E17 operator credit | H4 | does crossover drive best-sequence improvements, by source? |
| E18 blend coherence | H1 | are economic blends destructive vs their parents at the same BF? |
| E19 repair disruption | H2 | does repair rewrite crossover children in single/high-BF? |
| E20 elite-pool collapse | H3 | do the top-2 parents converge to near-clones faster in single? |
| E21 action composition | H5 | does the gap track combat-action availability, not the label? |

In [ ]:
!python -m pip install -q pandas matplotlib numpy scipy
!python analysis/analyze_v5_full.py test_results/v5

In [ ]:
import glob
from IPython.display import Image, display
for f in sorted(glob.glob('analysis/figures_v5_full/*.png')):
    print(f); display(Image(filename=f))

In [ ]:
import glob, pandas as pd
from IPython.display import display, Markdown
for f in sorted(glob.glob('analysis/tables_v5_full/*_summary.csv') + glob.glob('analysis/tables_v5_full/E*.csv')):
    display(Markdown(f'### {f}')); display(pd.read_csv(f, index_col=0))